Setup and imports

In [ ]:
import torch
import torchvision
from torchvision import transforms
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
import os

In [ ]:
RAND_SEED = 42
random.seed(RAND_SEED)
np.random.seed(RAND_SEED)
torch.manual_seed(RAND_SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Data exploration & binary label transformation

In [ ]:
df = pd.read_csv('../data/train.csv')
df.head(5)

In [ ]:
class_counts = df['diagnosis'].value_counts().sort_index()

ax = class_counts.plot(kind='bar', title='Class distribution', xlabel = 'DR Severity', ylabel='Occurences')
ax.bar_label(ax.containers[0], label_type='edge', padding=3)
ax.margins(y=0.1)

plt.show()

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
sample_rows = df.sample(5, random_state=RAND_SEED)

for ax, (_, row) in zip(axes, sample_rows.iterrows()):
    img_path = f"../data/train_images/{row['id_code']}.png"
    img = plt.imread(img_path)
    ax.imshow(img)
    ax.set_title(f"Severity: {row['diagnosis']}")
    ax.axis('off')
plt.show()

In [ ]:
#Reduce to binary labels
df['binary_label'] = (df['diagnosis'] > 0).astype(int)

print(df['binary_label'].value_counts())
print(f"\nClass ratio: {df['binary_label'].value_counts(normalize=True)}")

Train/test split

In [ ]:
from sklearn.model_selection import train_test_split

#Training data split
train_df, val_test_df = train_test_split(
    df, 
    test_size=0.3, 
    stratify=df['binary_label'], 
    random_state=RAND_SEED
)

#Test and validation data split
val_df, test_df = train_test_split(
    val_test_df, 
    test_size=0.5,
    stratify=val_test_df['binary_label'], 
    random_state=RAND_SEED
)

train_balance, val_balance, test_balance = train_df['binary_label'].value_counts(normalize=True), val_df['binary_label'].value_counts(normalize=True), test_df['binary_label'].value_counts(normalize=True)

print(f'Training set: {len(train_df)}, Validation set: {len(val_df)}, Test set: {len(test_df)}')
print(f'\nTrain class balance:\n{train_balance}')
print(f"\nVal class balance:\n{val_balance}")
print(f"\nTest class balance:\n{test_balance}")

Transform, Dataset and DataLoader

In [ ]:
from PIL import Image
from torch.utils.data import Dataset, DataLoader

MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15), #Flip and rotate add noise to prevent overfitting on training data.
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])

In [ ]:
class RetinaDataset(Dataset):
    def __init__(self, df, image_dir, transform):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = f"{self.image_dir}/{row['id_code']}.png"
        image = Image.open(img_path).convert('RGB')
        image = self.transform(image)
        label = torch.tensor(row['binary_label'], dtype=torch.float32)
        return image, label

In [ ]:
BATCH_SIZE = 32

train_dataset = RetinaDataset(train_df, '../data/train_images', train_transform)
val_dataset = RetinaDataset(val_df, '../data/train_images', eval_transform)
test_dataset = RetinaDataset(test_df, '../data/train_images', eval_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

In [ ]:
images, labels = next(iter(train_loader))
print(images.shape, labels.shape)

Model

In [ ]:
import torch.nn as nn
from torchvision import models

model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

for param in model.parameters():
    param.requires_grad = False #Freeze all layers before unfreezing final layer for fine tuning

for param in model.layer4.parameters():
    param.requires_grad = True

num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 1)

model = model.to(device)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable_params:,} / Total: {total_params:,}')

Training

In [ ]:
pos_count = train_df['binary_label'].sum()
neg_count = len(train_df) - pos_count
pos_weight = torch.tensor(neg_count / pos_count).to(device) #Not necessary here due to balanced class distribution, still useful, use later in 5-class CNN.
print(f'pos_weight: {pos_weight.item():.3f}')

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

In [ ]:
NUM_EPOCHS = 8
best_val_loss = float('inf')

for epoch in range(NUM_EPOCHS):
    model.train()
    running_train_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images).squeeze(1)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * images.size(0)

    train_loss = running_train_loss / len(train_dataset)

    model.eval()
    running_val_loss = 0.0
    correct = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images).squeeze(1)
            loss = criterion(outputs, labels)
            running_val_loss += loss.item() * images.size(0)

            preds = (torch.sigmoid(outputs) > 0.5).float()
            correct += (preds == labels).sum().item()

    val_loss = running_val_loss / len(val_dataset)
    val_acc = correct / len(val_dataset)

    print(f'Epoch {epoch+1}/{NUM_EPOCHS} | train_loss: {train_loss:.4f} | val_loss: {val_loss:.4f} | val_acc: {val_acc:.4f}')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), '../outputs/best_model.pth')
        print('  -> saved new best model')

Test set evalution

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

eval_model = models.resnet18(weights=None)
eval_model.fc = nn.Linear(eval_model.fc.in_features, 1)
eval_model.load_state_dict(torch.load('../outputs/best_model.pth', map_location=device))
eval_model = eval_model.to(device)
eval_model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = eval_model(images).squeeze(1)
        preds = (torch.sigmoid(outputs) > 0.5).float().cpu()

        all_preds.extend(preds.numpy())
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=['No DR', 'DR Present']))

cm = confusion_matrix(all_labels, all_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['No DR', 'DR Present'], yticklabels=['No DR', 'DR Present'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Test Set Confusion Matrix')
plt.show()